### Data preparation for curriculum experiment
1. Stage 1: All synthetic data
2. Stage 2: 30K real-upsampled + 30K synthetic
3. Stage 3: 1K real only with low learning rate

In [ ]:
from pathlib import Path
import pandas as pd
import random

OUTPUT_DIR = Path("sentence_pairs_data/curriculum")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)

# Synthetic train files
SYNTHETIC_ID_EN_TO_BBC_PATH = Path("sentence_pairs_data/synthetic_nllb_id_en_to_bbc_googletrans_clean.jsonl")
SYNTHETIC_BBC_TO_ID_EN_PATH = Path("sentence_pairs_data/synthetic_wikipedia_bbc_to_id_en_googletrans_clean.jsonl")

# NusaX train files
NUSAX_IND_TO_BBC_TRAIN_PATH = Path("sentence_pairs_data/nusax_ind_to_bbc_train.jsonl")
NUSAX_ENG_TO_BBC_TRAIN_PATH = Path("sentence_pairs_data/nusax_eng_to_bbc_train.jsonl")
NUSAX_BBC_TO_IND_TRAIN_PATH = Path("sentence_pairs_data/nusax_bbc_to_ind_train.jsonl")
NUSAX_BBC_TO_ENG_TRAIN_PATH = Path("sentence_pairs_data/nusax_bbc_to_eng_train.jsonl")

# NusaX validation files
NUSAX_IND_TO_BBC_VALIDATION_PATH = Path("sentence_pairs_data/nusax_ind_to_bbc_validation.jsonl")
NUSAX_ENG_TO_BBC_VALIDATION_PATH = Path("sentence_pairs_data/nusax_eng_to_bbc_validation.jsonl")
NUSAX_BBC_TO_IND_VALIDATION_PATH = Path("sentence_pairs_data/nusax_bbc_to_ind_validation.jsonl")
NUSAX_BBC_TO_ENG_VALIDATION_PATH = Path("sentence_pairs_data/nusax_bbc_to_eng_validation.jsonl")

# NusaX test files
NUSAX_IND_TO_BBC_TEST_PATH = Path("sentence_pairs_data/nusax_ind_to_bbc_test.jsonl")
NUSAX_ENG_TO_BBC_TEST_PATH = Path("sentence_pairs_data/nusax_eng_to_bbc_test.jsonl")
NUSAX_BBC_TO_IND_TEST_PATH = Path("sentence_pairs_data/nusax_bbc_to_ind_test.jsonl")
NUSAX_BBC_TO_ENG_TEST_PATH = Path("sentence_pairs_data/nusax_bbc_to_eng_test.jsonl")


# =========================
# Config
# =========================

REQUIRED_COLUMNS = ["src_lang", "tgt_lang", "source", "target"]

N_REAL_UPSAMPLED_STAGE2 = 30_000
N_SYNTHETIC_STAGE2 = 30_000

# Set this to True if you only want English -> Batak Toba
FILTER_ONE_DIRECTION = False
SRC_LANG = "eng"
TGT_LANG = "bbc"

LANG_MAP = {
    "eng_Latn": "eng",
    "en": "eng",
    "eng": "eng",

    "ind_Latn": "ind",
    "id": "ind",
    "ind": "ind",

    "bbc_Latn": "bbc",
    "bbc": "bbc",
}


# =========================
# Helpers
# =========================
def normalize_lang_code(lang: str) -> str:
    lang = str(lang).strip()

    if lang not in LANG_MAP:
        raise ValueError(f"Unknown language code: {lang}")

    return LANG_MAP[lang]

def load_jsonl(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")

    df = pd.read_json(path, lines=True)

    missing = [col for col in REQUIRED_COLUMNS if col not in df.columns]
    if missing:
        raise ValueError(f"{path} is missing columns: {missing}")

    df = df[REQUIRED_COLUMNS].copy()

    df = df.dropna(subset=REQUIRED_COLUMNS)

    for col in REQUIRED_COLUMNS:
        df[col] = df[col].astype(str).str.strip()

    # Normalize language codes
    df["src_lang"] = df["src_lang"].map(normalize_lang_code)
    df["tgt_lang"] = df["tgt_lang"].map(normalize_lang_code)

    df = df[(df["source"] != "") & (df["target"] != "")]

    df = df.drop_duplicates(
        subset=REQUIRED_COLUMNS,
        keep="first",
    )

    return df.reset_index(drop=True)


def save_jsonl(df: pd.DataFrame, path: Path):
    df = df[REQUIRED_COLUMNS].copy()
    df.to_json(
        path,
        orient="records",
        lines=True,
        force_ascii=False,
    )
    print(f"Saved {len(df):,} rows -> {path}")


def shuffle_df(df: pd.DataFrame, seed: int = SEED) -> pd.DataFrame:
    return df.sample(frac=1, random_state=seed).reset_index(drop=True)


def upsample_df(df: pd.DataFrame, n: int, seed: int = SEED) -> pd.DataFrame:
    if len(df) == 0:
        raise ValueError("Cannot upsample an empty dataframe.")

    return df.sample(n=n, replace=True, random_state=seed).reset_index(drop=True)


def sample_df(df: pd.DataFrame, n: int, seed: int = SEED) -> pd.DataFrame:
    if len(df) == 0:
        raise ValueError("Cannot sample from an empty dataframe.")

    replace = n > len(df)

    return df.sample(n=n, replace=replace, random_state=seed).reset_index(drop=True)


def maybe_filter_direction(df: pd.DataFrame) -> pd.DataFrame:
    if not FILTER_ONE_DIRECTION:
        return df.reset_index(drop=True)

    filtered = df[
        (df["src_lang"] == SRC_LANG) &
        (df["tgt_lang"] == TGT_LANG)
    ].copy()

    return filtered.reset_index(drop=True)


def print_lang_distribution(name: str, df: pd.DataFrame):
    print(f"\n{name}")
    print("-" * 40)
    print(f"Total rows: {len(df):,}")

    if len(df) > 0:
        print(df.groupby(["src_lang", "tgt_lang"]).size())


# =========================
# Load synthetic train data
# =========================

id_en_to_bbc_df = load_jsonl(SYNTHETIC_ID_EN_TO_BBC_PATH)
bbc_to_id_en_df = load_jsonl(SYNTHETIC_BBC_TO_ID_EN_PATH)

synthetic_df = pd.concat(
    [
        id_en_to_bbc_df,
        bbc_to_id_en_df,
    ],
    ignore_index=True,
)

synthetic_df = synthetic_df.drop_duplicates(
    subset=REQUIRED_COLUMNS,
    keep="first",
).reset_index(drop=True)

synthetic_df = maybe_filter_direction(synthetic_df)
synthetic_df = shuffle_df(synthetic_df, SEED)

print_lang_distribution("Synthetic train data", synthetic_df)


# =========================
# Load NusaX train data
# =========================

real_train_df = pd.concat(
    [
        load_jsonl(NUSAX_IND_TO_BBC_TRAIN_PATH),
        load_jsonl(NUSAX_ENG_TO_BBC_TRAIN_PATH),
        load_jsonl(NUSAX_BBC_TO_IND_TRAIN_PATH),
        load_jsonl(NUSAX_BBC_TO_ENG_TRAIN_PATH),
    ],
    ignore_index=True,
)

real_train_df = real_train_df.drop_duplicates(
    subset=REQUIRED_COLUMNS,
    keep="first",
).reset_index(drop=True)

real_train_df = maybe_filter_direction(real_train_df)
real_train_df = shuffle_df(real_train_df, SEED)

print_lang_distribution("Real NusaX train data", real_train_df)


# =========================
# Load NusaX validation data
# =========================

real_valid_df = pd.concat(
    [
        load_jsonl(NUSAX_IND_TO_BBC_VALIDATION_PATH),
        load_jsonl(NUSAX_ENG_TO_BBC_VALIDATION_PATH),
        load_jsonl(NUSAX_BBC_TO_IND_VALIDATION_PATH),
        load_jsonl(NUSAX_BBC_TO_ENG_VALIDATION_PATH),
    ],
    ignore_index=True,
)

real_valid_df = real_valid_df.drop_duplicates(
    subset=REQUIRED_COLUMNS,
    keep="first",
).reset_index(drop=True)

real_valid_df = maybe_filter_direction(real_valid_df)
real_valid_df = shuffle_df(real_valid_df, SEED)

print_lang_distribution("Real NusaX validation data", real_valid_df)


# =========================
# Load NusaX test data
# =========================

real_test_df = pd.concat(
    [
        load_jsonl(NUSAX_IND_TO_BBC_TEST_PATH),
        load_jsonl(NUSAX_ENG_TO_BBC_TEST_PATH),
        load_jsonl(NUSAX_BBC_TO_IND_TEST_PATH),
        load_jsonl(NUSAX_BBC_TO_ENG_TEST_PATH),
    ],
    ignore_index=True,
)

real_test_df = real_test_df.drop_duplicates(
    subset=REQUIRED_COLUMNS,
    keep="first",
).reset_index(drop=True)

real_test_df = maybe_filter_direction(real_test_df)
real_test_df = shuffle_df(real_test_df, SEED)

print_lang_distribution("Real NusaX test data", real_test_df)


# =========================
# Stage 1: synthetic only
# =========================

stage1_df = synthetic_df.copy()
stage1_df = shuffle_df(stage1_df, SEED)

save_jsonl(
    stage1_df,
    OUTPUT_DIR / "train_stage1_synthetic.jsonl",
)


# =========================
# Stage 2: real-upsampled + synthetic
# =========================

real_upsampled_df = upsample_df(
    real_train_df,
    n=N_REAL_UPSAMPLED_STAGE2,
    seed=SEED,
)

synthetic_sampled_df = sample_df(
    synthetic_df,
    n=N_SYNTHETIC_STAGE2,
    seed=SEED,
)

stage2_df = pd.concat(
    [
        real_upsampled_df,
        synthetic_sampled_df,
    ],
    ignore_index=True,
)

stage2_df = shuffle_df(stage2_df, SEED)

save_jsonl(
    stage2_df,
    OUTPUT_DIR / "train_stage2_mixed.jsonl",
)


# =========================
# Stage 3: real train only
# =========================

stage3_df = shuffle_df(real_train_df, SEED)

save_jsonl(
    stage3_df,
    OUTPUT_DIR / "train_stage3_real_only.jsonl",
)


# =========================
# Validation and test
# =========================

save_jsonl(
    real_valid_df,
    OUTPUT_DIR / "valid_real.jsonl",
)

save_jsonl(
    real_test_df,
    OUTPUT_DIR / "test_real.jsonl",
)


# =========================
# Final summary
# =========================

print("\nCurriculum dataset summary")
print("=" * 50)
print(f"Stage 1 synthetic:      {len(stage1_df):,}")
print(f"Stage 2 mixed:          {len(stage2_df):,}")
print(f"  - real upsampled:     {len(real_upsampled_df):,}")
print(f"  - synthetic sampled:  {len(synthetic_sampled_df):,}")
print(f"Stage 3 real only:      {len(stage3_df):,}")
print(f"Validation real:        {len(real_valid_df):,}")
print(f"Test real:              {len(real_test_df):,}")

print_lang_distribution("Stage 1 distribution", stage1_df)
print_lang_distribution("Stage 2 distribution", stage2_df)
print_lang_distribution("Stage 3 distribution", stage3_df)
print_lang_distribution("Validation distribution", real_valid_df)
print_lang_distribution("Test distribution", real_test_df)


Synthetic train data
----------------------------------------
Total rows: 74,634
src_lang  tgt_lang
bbc       ind          8503
eng       bbc         14180
ind       bbc         51951
dtype: int64

Real NusaX train data
----------------------------------------
Total rows: 2,000
src_lang  tgt_lang
bbc       eng         500
          ind         500
eng       bbc         500
ind       bbc         500
dtype: int64

Real NusaX validation data
----------------------------------------
Total rows: 400
src_lang  tgt_lang
bbc       eng         100
          ind         100
eng       bbc         100
ind       bbc         100
dtype: int64

Real NusaX test data
----------------------------------------
Total rows: 1,600
src_lang  tgt_lang
bbc       eng         400
          ind         400
eng       bbc         400
ind       bbc         400
dtype: int64
Saved 74,634 rows -> sentence_pairs_data/curriculum/train_stage1_synthetic.jsonl
Saved 60,000 rows -> sentence_pairs_data/curriculum/train_stage2_

### New proposal:
1. Stage 1: All synthetic data, cap to 100K. Balance the direction. ind/eng -> bbc 50%, bbc -> ind/eng 50%, and from bbc synthetic data put all in ind/eng -> bbc
2. Stage 2: 30K real-upsampled + 30K synthetic (ensure the synthetic is in ind/eng -> bbc)
3. Stage 3: 1K real only with low learning rate

1. Stage 1: All synthetic data, cap to 100k rows. For case A data real Batak → synthetic English/Indonesia use for training direction eng_Latn/ind_Latn → bbc_Latn, 60k rows. For case B data real English/Indonesia → synthetic Batak use for training direction bbc_Latn → eng_Latn/ind_Latn, 40k rows. Make the ratio of A and B adjustable for future experiments. Ratio of English/Indonesia keep 50:50. Pay attention to data real English/Indonesia → synthetic Batak, the ratio of English/Indonesia is 30:70 originally, so maybe maximize english and get enough Indonesia to fulfill the ratio become 50:50.
2. Stage 2: Mix of real-upsampled and synthetic, cap for 20k rows. Real-upsampled data is from nusaX upsample until 10k. Synthetic data is same as stage 1, however for this stage only pick data that not included in stage 1. It means when processing stage 1, reserve 10k for stage 2.
3. Stage 3: 2000 real with 4 direction, each direction has 500 pairs: eng->bbc, ind->bbc, bbc->eng, bbc->ind. Train with low learning rate

Final recipe:
1. Stage 1: 90k synthetic pairs
2. Stage 2: 20k total, 10k real and 10k synthetic.
3. Stage 3: 2k real

In [ ]:
from pathlib import Path
import hashlib
import random

import pandas as pd


# ============================================================
# Paths
# ============================================================

OUTPUT_DIR = Path("sentence_pairs_data/curriculum")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Synthetic train files
# Case B original: real English/Indonesia -> synthetic Batak
SYNTHETIC_ID_EN_TO_BBC_PATH = Path(
    "sentence_pairs_data/synthetic_nllb_id_en_to_bbc_googletrans_clean.jsonl"
)

# Case A original: real Batak -> synthetic English/Indonesia
SYNTHETIC_BBC_TO_ID_EN_PATH = Path(
    "sentence_pairs_data/synthetic_wikipedia_bbc_to_id_en_googletrans_clean.jsonl"
)

SYNTHETIC_BBC_TO_ID_EN_PATH_MADLAD = Path(
    "sentence_pairs_data/synthetic_madlad_bbc_to_id_en_googletrans.jsonl"
)

# NusaX train files
NUSAX_IND_TO_BBC_TRAIN_PATH = Path("sentence_pairs_data/nusax_ind_to_bbc_train.jsonl")
NUSAX_ENG_TO_BBC_TRAIN_PATH = Path("sentence_pairs_data/nusax_eng_to_bbc_train.jsonl")
NUSAX_BBC_TO_IND_TRAIN_PATH = Path("sentence_pairs_data/nusax_bbc_to_ind_train.jsonl")
NUSAX_BBC_TO_ENG_TRAIN_PATH = Path("sentence_pairs_data/nusax_bbc_to_eng_train.jsonl")

# NusaX validation files
NUSAX_IND_TO_BBC_VALIDATION_PATH = Path("sentence_pairs_data/nusax_ind_to_bbc_validation.jsonl")
NUSAX_ENG_TO_BBC_VALIDATION_PATH = Path("sentence_pairs_data/nusax_eng_to_bbc_validation.jsonl")
NUSAX_BBC_TO_IND_VALIDATION_PATH = Path("sentence_pairs_data/nusax_bbc_to_ind_validation.jsonl")
NUSAX_BBC_TO_ENG_VALIDATION_PATH = Path("sentence_pairs_data/nusax_bbc_to_eng_validation.jsonl")

# NusaX test files
NUSAX_IND_TO_BBC_TEST_PATH = Path("sentence_pairs_data/nusax_ind_to_bbc_test.jsonl")
NUSAX_ENG_TO_BBC_TEST_PATH = Path("sentence_pairs_data/nusax_eng_to_bbc_test.jsonl")
NUSAX_BBC_TO_IND_TEST_PATH = Path("sentence_pairs_data/nusax_bbc_to_ind_test.jsonl")
NUSAX_BBC_TO_ENG_TEST_PATH = Path("sentence_pairs_data/nusax_bbc_to_eng_test.jsonl")


# ============================================================
# Config
# ============================================================

SEED = 42
random.seed(SEED)

REQUIRED_COLUMNS = ["src_lang", "tgt_lang", "source", "target"]

# Final recipe
N_STAGE1_SYNTHETIC = 90_000
N_STAGE2_TOTAL = 20_000
N_STAGE2_REAL = 10_000
N_STAGE2_SYNTHETIC = 10_000
N_STAGE3_REAL = 2_000

# Synthetic ratio:
# Case A:
#   original real Batak -> synthetic English/Indonesia
#   training direction: English/Indonesia -> Batak
#
# Case B:
#   original real English/Indonesia -> synthetic Batak
#   training direction: Batak -> English/Indonesia
#
# Default final recipe:
#   total synthetic budget = 100k
#   Case A = 60k total synthetic
#   Case B = 40k total synthetic
CASE_A_RATIO = 0.60
CASE_B_RATIO = 0.40

assert abs((CASE_A_RATIO + CASE_B_RATIO) - 1.0) < 1e-9

# Keep English/Indonesian balanced inside each case
EN_RATIO_WITHIN_CASE = 0.50
ID_RATIO_WITHIN_CASE = 0.50

assert abs((EN_RATIO_WITHIN_CASE + ID_RATIO_WITHIN_CASE) - 1.0) < 1e-9

# Stage 2 real and Stage 3 real are balanced over 4 directions
REAL_DIRECTIONS = [
    ("eng", "bbc"),
    ("ind", "bbc"),
    ("bbc", "eng"),
    ("bbc", "ind"),
]

N_STAGE2_REAL_PER_DIRECTION = N_STAGE2_REAL // 4
N_STAGE3_REAL_PER_DIRECTION = N_STAGE3_REAL // 4

assert N_STAGE2_REAL_PER_DIRECTION * 4 == N_STAGE2_REAL
assert N_STAGE3_REAL_PER_DIRECTION * 4 == N_STAGE3_REAL

# Output language code style.
# Use True for NLLB-style codes: eng_Latn, ind_Latn, bbc_Latn.
# Use False for short codes: eng, ind, bbc.
SAVE_AS_NLLB_CODES = True

LANG_MAP = {
    "eng_Latn": "eng",
    "en": "eng",
    "eng": "eng",

    "ind_Latn": "ind",
    "id": "ind",
    "ind": "ind",

    "bbc_Latn": "bbc",
    "bbc": "bbc",
}

NLLB_LANG_MAP = {
    "eng": "eng_Latn",
    "ind": "ind_Latn",
    "bbc": "bbc_Latn",
}


# ============================================================
# Helpers
# ============================================================

def normalize_lang_code(lang: str) -> str:
    lang = str(lang).strip()

    if lang not in LANG_MAP:
        raise ValueError(f"Unknown language code: {lang}")

    return LANG_MAP[lang]


def to_output_lang_code(lang: str) -> str:
    lang = normalize_lang_code(lang)

    if SAVE_AS_NLLB_CODES:
        return NLLB_LANG_MAP[lang]

    return lang


def add_pair_id(df: pd.DataFrame) -> pd.DataFrame:
    """
    Stable internal ID for overlap checks.
    Not saved to output files.
    """
    df = df.copy()

    def make_hash(row):
        text = "|||".join(
            [
                str(row["src_lang"]),
                str(row["tgt_lang"]),
                str(row["source"]),
                str(row["target"]),
            ]
        )
        return hashlib.sha1(text.encode("utf-8")).hexdigest()

    df["_pair_id"] = df.apply(make_hash, axis=1)
    return df


def load_jsonl(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")

    df = pd.read_json(path, lines=True)

    missing = [col for col in REQUIRED_COLUMNS if col not in df.columns]
    if missing:
        raise ValueError(f"{path} is missing columns: {missing}")

    df = df[REQUIRED_COLUMNS].copy()
    df = df.dropna(subset=REQUIRED_COLUMNS)

    for col in REQUIRED_COLUMNS:
        df[col] = df[col].astype(str).str.strip()

    df["src_lang"] = df["src_lang"].map(normalize_lang_code)
    df["tgt_lang"] = df["tgt_lang"].map(normalize_lang_code)

    df = df[(df["source"] != "") & (df["target"] != "")]

    df = df.drop_duplicates(
        subset=REQUIRED_COLUMNS,
        keep="first",
    ).reset_index(drop=True)

    return df


def save_jsonl(df: pd.DataFrame, path: Path):
    df = df[REQUIRED_COLUMNS].copy()

    df["src_lang"] = df["src_lang"].map(to_output_lang_code)
    df["tgt_lang"] = df["tgt_lang"].map(to_output_lang_code)

    df.to_json(
        path,
        orient="records",
        lines=True,
        force_ascii=False,
    )

    print(f"Saved {len(df):,} rows -> {path}")


def shuffle_df(df: pd.DataFrame, seed: int = SEED) -> pd.DataFrame:
    return df.sample(frac=1, random_state=seed).reset_index(drop=True)


def reverse_direction(df: pd.DataFrame) -> pd.DataFrame:
    """
    Swap source/target and src_lang/tgt_lang.
    """
    reversed_df = pd.DataFrame(
        {
            "src_lang": df["tgt_lang"],
            "tgt_lang": df["src_lang"],
            "source": df["target"],
            "target": df["source"],
        }
    )

    reversed_df = reversed_df.drop_duplicates(
        subset=REQUIRED_COLUMNS,
        keep="first",
    ).reset_index(drop=True)

    return reversed_df


def filter_direction(df: pd.DataFrame, src_lang: str, tgt_lang: str) -> pd.DataFrame:
    return df[
        (df["src_lang"] == src_lang) &
        (df["tgt_lang"] == tgt_lang)
    ].copy().reset_index(drop=True)


def sample_df(df: pd.DataFrame, n: int, seed: int = SEED) -> pd.DataFrame:
    """
    Exact-size sampling.
    Uses replacement only if n > available rows.
    """
    if len(df) == 0:
        raise ValueError("Cannot sample from an empty dataframe.")

    replace = n > len(df)

    if replace:
        print(
            f"WARNING: sampling with replacement because requested {n:,} rows "
            f"but only found {len(df):,} rows."
        )

    return df.sample(
        n=n,
        replace=replace,
        random_state=seed,
    ).reset_index(drop=True)


def sample_by_direction(
    df: pd.DataFrame,
    direction_counts: dict,
    seed: int = SEED,
) -> pd.DataFrame:
    parts = []

    for i, ((src_lang, tgt_lang), n) in enumerate(direction_counts.items()):
        direction_df = filter_direction(df, src_lang, tgt_lang)

        if len(direction_df) == 0:
            raise ValueError(
                f"No rows found for direction {src_lang} -> {tgt_lang}"
            )

        sampled = sample_df(
            direction_df,
            n=n,
            seed=seed + i,
        )

        parts.append(sampled)

    return shuffle_df(pd.concat(parts, ignore_index=True), seed)


def split_count_by_ratio(total: int, ratio_first: float):
    first = int(round(total * ratio_first))
    second = total - first
    return first, second


def synthetic_counts(total: int) -> dict:
    """
    Return exact counts for synthetic directions.

    Case A:
      eng -> bbc
      ind -> bbc

    Case B:
      bbc -> eng
      bbc -> ind
    """
    n_case_a, n_case_b = split_count_by_ratio(total, CASE_A_RATIO)

    n_a_eng, n_a_ind = split_count_by_ratio(n_case_a, EN_RATIO_WITHIN_CASE)
    n_b_eng, n_b_ind = split_count_by_ratio(n_case_b, EN_RATIO_WITHIN_CASE)

    return {
        ("eng", "bbc"): n_a_eng,
        ("ind", "bbc"): n_a_ind,
        ("bbc", "eng"): n_b_eng,
        ("bbc", "ind"): n_b_ind,
    }


def disjoint_sample_stage1_and_stage2(
    df: pd.DataFrame,
    stage1_counts: dict,
    stage2_counts: dict,
    seed: int = SEED,
):
    """
    For each direction:
      - reserve Stage 2 synthetic rows first
      - remove them from Stage 1 candidate pool
      - sample Stage 1 from the remaining rows

    This guarantees no Stage 1 / Stage 2 synthetic overlap when there
    are enough unique rows for each direction.
    """
    stage1_parts = []
    stage2_parts = []

    for i, direction in enumerate(stage1_counts.keys()):
        src_lang, tgt_lang = direction
        n_stage1 = stage1_counts[direction]
        n_stage2 = stage2_counts[direction]

        direction_df = filter_direction(df, src_lang, tgt_lang)

        if len(direction_df) == 0:
            raise ValueError(
                f"No synthetic rows found for direction {src_lang} -> {tgt_lang}"
            )

        direction_df = shuffle_df(direction_df, seed + i)

        if "_pair_id" not in direction_df.columns:
            direction_df = add_pair_id(direction_df)

        # Reserve Stage 2 first.
        if len(direction_df) >= n_stage2:
            stage2_df = direction_df.iloc[:n_stage2].copy()
        else:
            print(
                f"WARNING: direction {src_lang}->{tgt_lang} has only "
                f"{len(direction_df):,} unique rows, less than Stage 2 reserve "
                f"{n_stage2:,}. Stage 2 will sample with replacement."
            )
            stage2_df = sample_df(direction_df, n_stage2, seed + 1000 + i)

        reserved_ids = set(stage2_df["_pair_id"].tolist())

        stage1_pool = direction_df[
            ~direction_df["_pair_id"].isin(reserved_ids)
        ].copy()

        if len(stage1_pool) == 0:
            print(
                f"WARNING: no non-overlapping Stage 1 pool left for "
                f"{src_lang}->{tgt_lang}. Overlap is unavoidable."
            )
            stage1_pool = direction_df.copy()

        stage1_df = sample_df(stage1_pool, n_stage1, seed + 2000 + i)

        stage1_parts.append(stage1_df)
        stage2_parts.append(stage2_df)

        print(
            f"Synthetic {src_lang}->{tgt_lang}: "
            f"Stage 1 = {len(stage1_df):,}, "
            f"Stage 2 reserve = {len(stage2_df):,}, "
            f"available unique = {len(direction_df):,}"
        )

    stage1_synthetic_df = shuffle_df(
        pd.concat(stage1_parts, ignore_index=True),
        seed,
    )

    stage2_synthetic_df = shuffle_df(
        pd.concat(stage2_parts, ignore_index=True),
        seed,
    )

    # Check synthetic overlap.
    overlap = set(stage1_synthetic_df["_pair_id"]) & set(stage2_synthetic_df["_pair_id"])

    if len(overlap) > 0:
        print(
            f"WARNING: Stage 1 and Stage 2 synthetic overlap found: "
            f"{len(overlap):,} pair IDs."
        )
    else:
        print("No Stage 1 / Stage 2 synthetic overlap found.")

    return stage1_synthetic_df, stage2_synthetic_df


def print_lang_distribution(name: str, df: pd.DataFrame):
    print(f"\n{name}")
    print("-" * 50)
    print(f"Total rows: {len(df):,}")

    if len(df) > 0:
        print(df.groupby(["src_lang", "tgt_lang"]).size())


# ============================================================
# Load and transform synthetic data
# ============================================================

# ------------------------------------------------------------
# Case A
# Original:
#   real Batak -> synthetic English/Indonesia
#
# Input direction:
#   bbc -> eng / ind
#
# Training direction wanted:
#   eng / ind -> bbc
#
# Therefore: reverse direction.
# ------------------------------------------------------------

case_a_wikipedia_df = load_jsonl(SYNTHETIC_BBC_TO_ID_EN_PATH)
case_a_madlad_df = load_jsonl(SYNTHETIC_BBC_TO_ID_EN_PATH_MADLAD)

case_a_original_df = pd.concat(
    [
        case_a_wikipedia_df,
        case_a_madlad_df,
    ],
    ignore_index=True,
)

case_a_original_df = case_a_original_df.drop_duplicates(
    subset=REQUIRED_COLUMNS,
    keep="first",
).reset_index(drop=True)

case_a_train_df = reverse_direction(case_a_original_df)

# Keep only eng/ind -> bbc
case_a_train_df = case_a_train_df[
    (
        ((case_a_train_df["src_lang"] == "eng") | (case_a_train_df["src_lang"] == "ind")) &
        (case_a_train_df["tgt_lang"] == "bbc")
    )
].copy().reset_index(drop=True)

case_a_train_df["case"] = "A"


# ------------------------------------------------------------
# Case B
# Original:
#   real English/Indonesia -> synthetic Batak
#
# Input direction:
#   eng / ind -> bbc
#
# Training direction wanted:
#   bbc -> eng / ind
#
# Therefore: reverse direction.
# ------------------------------------------------------------

case_b_original_df = load_jsonl(SYNTHETIC_ID_EN_TO_BBC_PATH)

case_b_train_df = reverse_direction(case_b_original_df)

# Keep only bbc -> eng/ind
case_b_train_df = case_b_train_df[
    (
        (case_b_train_df["src_lang"] == "bbc") &
        ((case_b_train_df["tgt_lang"] == "eng") | (case_b_train_df["tgt_lang"] == "ind"))
    )
].copy().reset_index(drop=True)

case_b_train_df["case"] = "B"


# ------------------------------------------------------------
# Combine synthetic
# ------------------------------------------------------------

synthetic_df = pd.concat(
    [
        case_a_train_df,
        case_b_train_df,
    ],
    ignore_index=True,
)

synthetic_df = synthetic_df[REQUIRED_COLUMNS + ["case"]].copy()

synthetic_df = synthetic_df.drop_duplicates(
    subset=REQUIRED_COLUMNS,
    keep="first",
).reset_index(drop=True)

synthetic_df = add_pair_id(synthetic_df)
synthetic_df = shuffle_df(synthetic_df, SEED)

print_lang_distribution("Synthetic data after direction transformation", synthetic_df)


# ============================================================
# Load real NusaX data
# ============================================================

real_train_df = pd.concat(
    [
        load_jsonl(NUSAX_ENG_TO_BBC_TRAIN_PATH),
        load_jsonl(NUSAX_IND_TO_BBC_TRAIN_PATH),
        load_jsonl(NUSAX_BBC_TO_ENG_TRAIN_PATH),
        load_jsonl(NUSAX_BBC_TO_IND_TRAIN_PATH),
    ],
    ignore_index=True,
)

real_train_df = real_train_df.drop_duplicates(
    subset=REQUIRED_COLUMNS,
    keep="first",
).reset_index(drop=True)

real_train_df = add_pair_id(real_train_df)
real_train_df = shuffle_df(real_train_df, SEED)

print_lang_distribution("Real NusaX train data", real_train_df)


real_valid_df = pd.concat(
    [
        load_jsonl(NUSAX_ENG_TO_BBC_VALIDATION_PATH),
        load_jsonl(NUSAX_IND_TO_BBC_VALIDATION_PATH),
        load_jsonl(NUSAX_BBC_TO_ENG_VALIDATION_PATH),
        load_jsonl(NUSAX_BBC_TO_IND_VALIDATION_PATH),
    ],
    ignore_index=True,
)

real_valid_df = real_valid_df.drop_duplicates(
    subset=REQUIRED_COLUMNS,
    keep="first",
).reset_index(drop=True)

real_valid_df = add_pair_id(real_valid_df)
real_valid_df = shuffle_df(real_valid_df, SEED)

print_lang_distribution("Real NusaX validation data", real_valid_df)


real_test_df = pd.concat(
    [
        load_jsonl(NUSAX_ENG_TO_BBC_TEST_PATH),
        load_jsonl(NUSAX_IND_TO_BBC_TEST_PATH),
        load_jsonl(NUSAX_BBC_TO_ENG_TEST_PATH),
        load_jsonl(NUSAX_BBC_TO_IND_TEST_PATH),
    ],
    ignore_index=True,
)

real_test_df = real_test_df.drop_duplicates(
    subset=REQUIRED_COLUMNS,
    keep="first",
).reset_index(drop=True)

real_test_df = add_pair_id(real_test_df)
real_test_df = shuffle_df(real_test_df, SEED)

print_lang_distribution("Real NusaX test data", real_test_df)


# ============================================================
# Stage 1 and Stage 2 synthetic split
# ============================================================

stage1_synthetic_counts = synthetic_counts(N_STAGE1_SYNTHETIC)
stage2_synthetic_counts = synthetic_counts(N_STAGE2_SYNTHETIC)

print("\nStage 1 synthetic target counts")
print("-" * 50)
for direction, count in stage1_synthetic_counts.items():
    print(f"{direction[0]} -> {direction[1]}: {count:,}")

print("\nStage 2 synthetic reserve target counts")
print("-" * 50)
for direction, count in stage2_synthetic_counts.items():
    print(f"{direction[0]} -> {direction[1]}: {count:,}")

stage1_df, stage2_synthetic_df = disjoint_sample_stage1_and_stage2(
    synthetic_df,
    stage1_counts=stage1_synthetic_counts,
    stage2_counts=stage2_synthetic_counts,
    seed=SEED,
)

stage1_df = shuffle_df(stage1_df, SEED)


# ============================================================
# Stage 2 real-upsampled
# ============================================================

stage2_real_counts = {
    direction: N_STAGE2_REAL_PER_DIRECTION
    for direction in REAL_DIRECTIONS
}

stage2_real_df = sample_by_direction(
    real_train_df,
    direction_counts=stage2_real_counts,
    seed=SEED,
)

stage2_df = pd.concat(
    [
        stage2_real_df,
        stage2_synthetic_df,
    ],
    ignore_index=True,
)

stage2_df = shuffle_df(stage2_df, SEED)

assert len(stage2_real_df) == N_STAGE2_REAL
assert len(stage2_synthetic_df) == N_STAGE2_SYNTHETIC
assert len(stage2_df) == N_STAGE2_TOTAL


# ============================================================
# Stage 3 real only
# ============================================================

stage3_real_counts = {
    direction: N_STAGE3_REAL_PER_DIRECTION
    for direction in REAL_DIRECTIONS
}

stage3_df = sample_by_direction(
    real_train_df,
    direction_counts=stage3_real_counts,
    seed=SEED,
)

stage3_df = shuffle_df(stage3_df, SEED)

assert len(stage3_df) == N_STAGE3_REAL


# ============================================================
# Save files
# ============================================================

save_jsonl(
    stage1_df,
    OUTPUT_DIR / "train_stage1_synthetic.jsonl",
)

save_jsonl(
    stage2_df,
    OUTPUT_DIR / "train_stage2_mixed.jsonl",
)

save_jsonl(
    stage3_df,
    OUTPUT_DIR / "train_stage3_real_only.jsonl",
)

save_jsonl(
    real_valid_df,
    OUTPUT_DIR / "valid_real.jsonl",
)

save_jsonl(
    real_test_df,
    OUTPUT_DIR / "test_real.jsonl",
)


# ============================================================
# Final summary
# ============================================================

print("\nCurriculum dataset summary")
print("=" * 60)
print(f"Stage 1 synthetic:          {len(stage1_df):,}")
print(f"Stage 2 mixed:              {len(stage2_df):,}")
print(f"  - real upsampled:         {len(stage2_real_df):,}")
print(f"  - synthetic reserved:     {len(stage2_synthetic_df):,}")
print(f"Stage 3 real only:          {len(stage3_df):,}")
print(f"Validation real:            {len(real_valid_df):,}")
print(f"Test real:                  {len(real_test_df):,}")

print_lang_distribution("Stage 1 distribution", stage1_df)
print_lang_distribution("Stage 2 distribution", stage2_df)
print_lang_distribution("Stage 2 real distribution", stage2_real_df)
print_lang_distribution("Stage 2 synthetic distribution", stage2_synthetic_df)
print_lang_distribution("Stage 3 distribution", stage3_df)
print_lang_distribution("Validation distribution", real_valid_df)
print_lang_distribution("Test distribution", real_test_df)